# 🏸 Badminton AI Studio trên Kaggle GPU (Tesla T4)
Chạy toàn bộ hệ thống Web + AI Tracking (YOLO11n-Pose + TrackNetV3) trên GPU miễn phí của Kaggle.

### ⚙️ Hướng dẫn cài đặt cấu hình Kaggle (Rất quan trọng):
1. Ở menu bên phải Kaggle (Notebook Options):
   - **Accelerator**: Chọn **GPU T4 x2** (hoặc GPU P100)
   - **Internet**: Chọn **Internet On** (bắt buộc bật để tải model và mở link web)
2. Bấm nút **Run All** (hoặc Shift+Enter chạy từng cell).
3. Nhấp vào đường link `https://xxx.trycloudflare.com` ở Cell cuối cùng để mở Web AI Studio và dùng trực tiếp trên trình duyệt!

In [ ]:
# Cell 1: Cài đặt các thư viện cần thiết
!pip install -q ultralytics fastapi uvicorn python-multipart pycloudflared pyyaml scipy pydantic
!apt-get update -qq && apt-get install -y -qq nodejs npm ffmpeg wget
print('✅ Cài đặt môi trường hoàn tất!')

In [ ]:
# Cell 2: Tải trọng số mô hình AI (YOLO11n-Pose + TrackNetV3)
import os
os.makedirs('weights', exist_ok=True)
!wget -q -nc -O weights/yolo11n-pose.pt https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n-pose.pt
!wget -q -nc -O weights/tracknetv3.pt https://huggingface.co/rahulanantharama/TrackNetV3-badminton/resolve/main/TrackNet_best.pt
print('✅ Đã tải xong trọng số AI vào thư mục weights/')

In [ ]:
# Cell 3: Tải mã nguồn mới nhất từ GitHub
!rm -rf BadmintonTracking_repo
!git clone -q https://github.com/evintran1508/BadmintonTracking.git BadmintonTracking_repo
!cp -rf BadmintonTracking_repo/* .
!rm -rf BadmintonTracking_repo
print('✅ Đã cập nhật mã nguồn mới nhất từ GitHub!')

In [ ]:
# Cell 4: Khởi chạy Web Server & Mở đường link Cloudflare Tunnel ra internet
import subprocess, time, sys

# Tự động cài đặt nếu phiên Kaggle vừa bị reset (tránh lỗi No module named 'pycloudflared')
try:
    from pycloudflared import try_cloudflare
except ImportError:
    print('🔄 Đang tự động cài bổ sung pycloudflared & thư viện...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pycloudflared', 'ultralytics', 'pydantic', 'pyyaml', 'scipy'])
    from pycloudflared import try_cloudflare

# Kiểm tra Node.js
try:
    subprocess.run(['node', '-v'], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
except Exception:
    print('🔄 Đang cài đặt Node.js...')
    subprocess.run('apt-get update -qq && apt-get install -y -qq nodejs npm ffmpeg', shell=True)

# Chạy server web Node.js ở cổng 3000
p = subprocess.Popen(['node', 'frontend/server.mjs'])
time.sleep(3)

# Mở Cloudflare Tunnel kết nối tới cổng 3000 (Miễn phí, không cần đăng ký)
print('⏳ Đang tạo đường link Cloudflare Tunnel...')
tunnel = try_cloudflare(port=3000)

print('\n' + '='*70)
print('🏸 BADMINTON AI STUDIO ĐANG CHẠY TRÊN KAGGLE GPU!')
print(f'👉 HÃY CLICK VÀO LINK NÀY ĐỂ MỞ WEB: {tunnel.tunnel}')
print('='*70 + '\n')

# Giữ tiến trình chạy để web hoạt động
try:
    p.wait()
except KeyboardInterrupt:
    p.terminate()
    print('Đã dừng server.')